In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os, sys

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_features_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
features_schema = get_features_schema() 

from superstore_logger import get_superstore_logger, log_event
from superstore_backfill_utils import is_dry_run
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_features_customers_products = get_superstore_logger("superstore_features_customers_products")


# -----------------------------
# Dry run
# -----------------------------
# The serving layer rebuilds from Gold and has no run-mode logic of its own —
# a replay or backfill simply produces different Gold for it to read. But it
# does write, so it must not write during a dry run. is_dry_run reads only the
# dry_run parameter and never raises, so a serving task cannot fail on the
# full_refresh gate that is enforced upstream where it belongs.
if is_dry_run(dbutils):
    log_event(
        logger_features_customers_products,
        "INFO",
        "Dry run: skipping features_customer_product write",
        dry_run=True,
        layer="Feature Engineering"
    )
    dbutils.notebook.exit("DRY_RUN_COMPLETED layer=features_customer_product (no data written)")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_features_customers_products,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="Feature Engineering"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
required_tables = [
    f"{catalog}.{gold_schema}.facts_orders",
    f"{catalog}.{gold_schema}.facts_sales"
]

missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, gold_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_features_customers_products, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create feature table.", layer="Feature Engineering")
else:
    # -------------------------------
    # Create or update feature customer_product Table (Environment-aware)
    # -------------------------------
    # Check if the customer_product feature table already exists
    if spark.catalog.tableExists(f"{catalog}.{features_schema}.features_customer_product"):
        # Table exists, perform MERGE to update existing rows and insert new ones
        spark.sql(f"""
            MERGE INTO {catalog}.{features_schema}.features_customer_product AS target
            USING (
                -- Aggregate customer-product metrics from Gold layer
                SELECT
                    o.customer_id,
                    s.product_id,

                    COUNT(DISTINCT s.order_id) AS purchase_count,           -- Total orders per customer-product
                    SUM(s.sales) AS total_spent,                            -- Total sales amount
                    SUM(s.profit) AS total_profit,                          -- Total profit
                    MAX(o.order_date) AS last_purchase_date,               -- Date of last purchase
                    DATEDIFF(current_date(), MAX(o.order_date)) AS days_since_last_purchase,  -- Recency
                    AVG(s.sales) AS avg_sales_per_order,                   -- Average sales per order
                    COUNT(DISTINCT CASE WHEN o.order_date >= DATE_SUB(current_date(), 30) 
                        THEN s.order_id END) AS orders_last_30d            -- Orders in last 30 days

                FROM {catalog}.{gold_schema}.facts_sales s
                JOIN {catalog}.{gold_schema}.facts_orders o
                    ON s.order_id = o.order_id

                GROUP BY 
                    o.customer_id, 
                    s.product_id
            ) AS source
            ON target.customer_id = source.customer_id
            AND target.product_id = source.product_id

            -- Update existing records if matched
            WHEN MATCHED THEN
                UPDATE SET
                    target.purchase_count = source.purchase_count,
                    target.total_spent = source.total_spent,
                    target.total_profit = source.total_profit,
                    target.last_purchase_date = source.last_purchase_date,
                    target.days_since_last_purchase = source.days_since_last_purchase,
                    target.avg_sales_per_order = source.avg_sales_per_order,
                    target.orders_last_30d = source.orders_last_30d

            -- Insert new records if not matched
            WHEN NOT MATCHED THEN
                INSERT (customer_id, product_id, purchase_count, total_spent, total_profit, last_purchase_date,
                        days_since_last_purchase, avg_sales_per_order, orders_last_30d)
                VALUES (source.customer_id, source.product_id, source.purchase_count, source.total_spent,
                        source.total_profit, source.last_purchase_date, source.days_since_last_purchase,
                        source.avg_sales_per_order, source.orders_last_30d)
        """)
    else:
        # Table does not exist, create it from scratch
        spark.sql(f"""
            -- Create the Customer-Product ML Features Table
            CREATE TABLE {catalog}.{features_schema}.features_customer_product AS
            SELECT
                o.customer_id,
                s.product_id,

                COUNT(DISTINCT s.order_id) AS purchase_count,
                SUM(s.sales) AS total_spent,
                SUM(s.profit) AS total_profit,
                MAX(o.order_date) AS last_purchase_date,
                DATEDIFF(current_date(), MAX(o.order_date)) AS days_since_last_purchase,
                AVG(s.sales) AS avg_sales_per_order,
                COUNT(DISTINCT CASE WHEN o.order_date >= DATE_SUB(current_date(), 30) THEN s.order_id END) AS orders_last_30d

            FROM {catalog}.{gold_schema}.facts_sales s
            JOIN {catalog}.{gold_schema}.facts_orders o
                ON s.order_id = o.order_id

            GROUP BY 
                o.customer_id, 
                s.product_id
        """)

log_event(logger_features_customers_products, "INFO", "Feature customer_product table created successfully.", layer="Feature Engineering")

